# Predicting late deliveries
Question: can we predict `late_delivery` at dispatch time?

Design (per checklist.md):
- Every consignment appears once, and a model would be used on future dispatches, so the **test set is the latest 20% of rows by `dispatched_on`**.
- Cross-validation inside the training rows uses expanding-window `TimeSeriesSplit`.
- `claim_paid_eur` is a claim settled *after* delivery (about 0 for on-time rows), so it is not known at prediction time and is **excluded**. `consignment` is an identifier and is dropped. `dispatched_on` is used for ordering, plus hour/weekday, which are known at dispatch.
- Preprocessing lives in a `Pipeline`, so it is fit inside each fold.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
from sklearn.base import clone

df = pd.read_csv('data.csv', parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
df['hour'] = df.dispatched_on.dt.hour
df['weekday'] = df.dispatched_on.dt.weekday
print(df.shape, df.late_delivery.mean())

## 1. Hold out the test set (latest 20% of rows)

In [ ]:
cut = int(len(df) * 0.8)
train, test = df.iloc[:cut].copy(), df.iloc[cut:].copy()
print(train.dispatched_on.min(), train.dispatched_on.max())
print(test.dispatched_on.min(), test.dispatched_on.max())
assert train.dispatched_on.max() <= test.dispatched_on.min()
# test is not touched again until section 5
FEATURES = ['carrier', 'distance_km', 'weight_kg', 'parcels', 'hour', 'weekday']
CAT, NUM = ['carrier'], ['distance_km', 'weight_kg', 'parcels', 'hour', 'weekday']
X, y = train[FEATURES], train.late_delivery
print('train late rate', y.mean())

## 2. Candidate models (named before any scoring)
1. `baseline`: predicts the training prior (trivial).
2. `logreg_C0.1`, `logreg_C1`, `logreg_C10`: logistic regression on scaled numeric + one-hot carrier.
3. `hgb_shallow`, `hgb_deep`: histogram gradient boosting (max_depth 3 / None, lr 0.05, 200 iters).

Selection metric: mean out-of-fold ROC AUC (threshold-free). The decision threshold is then chosen on pooled out-of-fold predictions to maximise F1 of the late class.

In [ ]:
def prep():
    return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), CAT),
                              ('n', StandardScaler(), NUM)])
def hgb(depth):
    return Pipeline([('p', ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), CAT), ('n', 'passthrough', NUM)])),
                     ('m', HistGradientBoostingClassifier(max_depth=depth, learning_rate=0.05, max_iter=200, random_state=0))])
CANDIDATES = {
    'baseline': Pipeline([('p', prep()), ('m', DummyClassifier(strategy='prior'))]),
    'logreg_C0.1': Pipeline([('p', prep()), ('m', LogisticRegression(C=0.1, max_iter=1000))]),
    'logreg_C1': Pipeline([('p', prep()), ('m', LogisticRegression(C=1, max_iter=1000))]),
    'logreg_C10': Pipeline([('p', prep()), ('m', LogisticRegression(C=10, max_iter=1000))]),
    'hgb_shallow': hgb(3),
    'hgb_deep': hgb(None),
}

## 3. Time-ordered cross-validation on training rows only

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
rows, oof = [], {}
for name, est in CANDIDATES.items():
    p = np.full(len(X), np.nan)
    aucs, aps = [], []
    for tr, va in tscv.split(X):
        m = clone(est).fit(X.iloc[tr], y.iloc[tr])
        p[va] = m.predict_proba(X.iloc[va])[:, 1]
        aucs.append(roc_auc_score(y.iloc[va], p[va])); aps.append(average_precision_score(y.iloc[va], p[va]))
    oof[name] = p
    rows.append((name, np.mean(aucs), np.std(aucs), np.mean(aps)))
cv = pd.DataFrame(rows, columns=['model', 'cv_auc', 'cv_auc_sd', 'cv_ap']).sort_values('cv_auc', ascending=False)
cv

In [ ]:
best = cv.iloc[0].model
mask = ~np.isnan(oof[best])
ths = np.linspace(0.05, 0.8, 76)
f1s = [f1_score(y[mask], oof[best][mask] >= t) for t in ths]
THRESH = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(THRESH, 2), '| CV F1:', round(max(f1s), 3))
print('CV F1 at 0.5:', round(f1_score(y[mask], oof[best][mask] >= 0.5), 3))

## 4. Choice, written down before touching the test set
Based on cross-validation only:
- **Model:** `logreg_C0.1` (highest mean out-of-fold AUC, 0.722; the three logistic variants are within noise of each other, fold SD about 0.02, and all beat the gradient boosting models and the baseline at 0.5).
- **Features:** carrier, distance_km, weight_kg, parcels, hour, weekday.
- **Threshold:** 0.24, which maximises pooled out-of-fold F1 (0.475). At the default 0.5 the CV F1 is only 0.27, because only about 21% of rows are late.

This choice is final; the test set is scored once below.

## 5. Refit on all training rows, score the test set once (headline)

In [ ]:
final = clone(CANDIDATES[best]).fit(X, y)
Xt, yt = test[FEATURES], test.late_delivery
pt = final.predict_proba(Xt)[:, 1]
base = clone(CANDIDATES['baseline']).fit(X, y).predict_proba(Xt)[:, 1]
pred = pt >= THRESH
print(f'test rows: {len(test)}, late rate: {yt.mean():.3f}')
print(f'ROC AUC  model {roc_auc_score(yt, pt):.3f} | baseline {roc_auc_score(yt, base):.3f}')
print(f'Avg precision model {average_precision_score(yt, pt):.3f} | baseline (prevalence) {yt.mean():.3f}')
print(f'F1 at threshold {THRESH:.2f}: {f1_score(yt, pred):.3f}')
print(f'Brier model {brier_score_loss(yt, pt):.3f} | baseline {brier_score_loss(yt, base):.3f}')
print(pd.crosstab(yt, pred, rownames=['actual'], colnames=['predicted late']))

In [ ]:
coef = pd.Series(final.named_steps['m'].coef_[0], index=final.named_steps['p'].get_feature_names_out()).sort_values()
coef

## 6. Conclusion and caveats
**Headline (test set, scored once):** ROC AUC 0.704 (baseline 0.5), average precision 0.069 (prevalence 0.022), F1 0.078 at threshold 0.24. The ranking signal from distance, weight and carrier (longer, heavier, coastal consignments are later) carries over to the later rows (CV AUC 0.72), but **the decision rule does not**.

**Prevalence shift.** The late rate is about 25% in the training rows but 2.2% in the test period. After the fact, a descriptive look at the late rate by week shows it falling to roughly 1-3% from October 2025 onward, in every carrier; earlier it moved between 5% and 55% by season. The test set falls entirely in that low regime, so the threshold of 0.24 tuned on CV flags 607 of 1611 consignments to catch 25 of 35 late ones (precision about 4%), and the Brier score is worse than the baseline (0.088 vs 0.077), meaning the probabilities are miscalibrated for the new period. The shift was seen only after scoring, so no choice was changed in response; it is reported as is.

**So:** we can rank consignments by lateness risk modestly well (AUC about 0.70), but with the features available at dispatch we cannot reliably give calibrated probabilities or a fixed alert threshold; they need to be recalibrated on recent data, and the cause of the drop in lateness (not in these columns) should be found. With only 35 late test cases, the test metrics are also noisy.

Note: `claim_paid_eur` would predict `late_delivery` almost perfectly, but only because it is paid after a late delivery, so it was excluded.